## Aquire more Training data from German Wikipedia

Load dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("wikimedia/wikipedia", "20231101.de")

ds

In [ ]:
all_sentences = []

## Load spacy and clean the text

First we `clean` the text from unwanted characters.

Then we use Spacy to chop the article texts into individual `sentences`.

In [ ]:
import re
import spacy

nlp = spacy.load("de_core_news_sm")


def clean_text(text):
    text = re.sub(r"[^\w\s.,!?']", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
for idx, item in enumerate(ds["train"]):
    if idx < 50: # start by index 50
        continue
    if len(all_sentences) >= 10000:
        print(idx, "articles processed, 10,000 sentences collected.")
        break

    article = item["text"]
    cleaned_article = clean_text(article)
    doc = nlp(cleaned_article)
    for sent in doc.sents:
        if len(sent.text.split()) > 4:
            all_sentences.append(sent.text)

In [ ]:
len(all_sentences)

In [ ]:
# Use this code to save the cleaned sentences into it's own file
# Then you can run the filter.py script on the file individually 

# import pandas as pd

# df = pd.DataFrame(all_sentences, columns=["text"])
# df.to_json("german_sentences.jsonl", orient="records", lines=True, force_ascii=False)

## Create an Ollama client

It is needed to for the `filter.py` script. We also load the env variables for the `OLLAMA_API_KEY`

In [ ]:
from ollama import Client
import os
from dotenv import load_dotenv

load_dotenv() 

client = Client(
            host="https://ollama.com",
            headers={'Authorization': 'Bearer ' + os.environ.get('OLLAMA_API_KEY')}
        )


## Filter out rows that are not quality data

Using the `filter.py` functions we will remove rows that are gibberish. This script uses GPT-OSS-120b and works in batches.

In [ ]:
import json
from src.utils import process_in_batches
from src.data.filter import filter_sentences

print(f"Starting to filter {len(all_sentences)} rows...")

for current_batch in process_in_batches(all_sentences, batch_size=10):
    filter_results = filter_sentences(current_batch, client)
    batch_result = [row for row in filter_results if row["keep"]]  # Keep only sentences marked for retention

    with open("../data/interim/german_sentences_filtered2.jsonl", "a", encoding="utf-8") as f:
        for line in batch_result:
            f.write(json.dumps(line, ensure_ascii=False) + "\n")